# 3 Build a Generic Ingestion Framework
# 4 Design a Common Data Model

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, lower, concat, lpad, lit, when, trim
from pyspark.sql.types import StringType, BooleanType
from pyspark.sql.dataframe import DataFrame
import re
import time
import json

#The second line (with 4g) specifies how much RAM to use. change according to machine
spark = SparkSession.builder \
    .appName("IngestionFramework") \
    .config("spark.driver.memory", "4g") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

In [16]:
# DATASET_CONFIG is a json file 
# Has entries that are specific to each dataset, like:
# primary_keys: if its a list then its the combination of these columns
# timestamp_cols: dict with columns cast to TimestampType (after snake_case rename).
# assembled_timestamp : dict with keys year_col, month_col, day_col,
#                         and optionally hour_col, minute_col, second_col,
#                         plus output_col for the new timestamp column name.
#                         Used when date parts live in separate integer columns.
# date_time_pairs     : list of {date_col, time_col, output_col, format}
#                         Used when date and time live in separate string columns.
# bool_cols           : list of column names to cast to BooleanType.
#                         Understands Y/N, yes/no, true/false, 1/0.
with open('dataset_config.json', 'r') as file:
    DATASET_CONFIGS = json.load(file)


    TYPE_MAP = {
        "integer": "int",
        "int": "int",
        "long": "bigint",
        "bigint": "bigint",
        "float": "float",
        "double": "double",
        "string": "string",
        "boolean": "boolean",
        "bool": "boolean",
        "timestamp": "timestamp",
        "date": "date",
    }

#-----------------------------Helpers-------------------

#column name to snake case

def to_snake_case(name):
    s = name.strip()
    s = re.sub(r'[\s\-]+', '_', s)                     # Replace spaces/dashes with single underscore first
    s1 = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1_\2', s)   # Handle runs of caps (e.g. PULocation -> PU_Location)
    s2 = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s1)    # Handle lower to upper transitions
    s3 = re.sub(r'_+', '_', s2)                        # Collapse any consecutive underscores
    return s3.lower()

#also snake case function
def standardize_columns(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, to_snake_case(c))
    return df


#validate if the rows in dataset are the same as expected_rows in config file (same name and type)
def validate_input_schema(df, config, dataset_name):
    errors = []
    actual_types = dict(df.dtypes)
    actual_cols = set(df.columns)
    declared_schema = config.get("schema", {})
    
    #validate column presence
    expected_cols = {to_snake_case(c) for c in declared_schema.keys()}  
    if missing := expected_cols - actual_cols:
        errors.append(f"Missing columns: {sorted(missing)}")
    if extra := actual_cols - expected_cols:
        errors.append(f"Extra columns: {sorted(extra)}")

    # validate data type
    for col_name, exp_type in config.get("schema", {}).items():
        act_type = actual_types.get(col_name)
        if not act_type:
            continue 
            
        
        exp_norm = TYPE_MAP.get(exp_type.lower(), exp_type.lower())
        act_norm = TYPE_MAP.get(act_type.lower(), act_type.lower())
        
        if exp_norm != act_norm:
            errors.append(f"Type mismatch '{col_name}': expected '{exp_norm}', got '{act_norm}'")

    if errors:
        raise ValueError(f"[{dataset_name}] Schema validation failed:\n - " + "\n - ".join(errors))

    
# Cast each column to the type declared in config["schema"]
def apply_schema(df, config):
    declared_schema = config.get("schema")
    if not declared_schema:
        return df
    for col_name, spark_type in declared_schema.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, col(col_name).cast(spark_type))
    return df


# Timestamp normalisation to format yyyy-mm-dd HH:mm:ss
#3 types of normalization

# Cast timestamp to TimeStampType
def normalize_timestamps(df, timestamp_cols):
    
    for col_name, fmt in timestamp_cols.items():
        snake = to_snake_case(col_name)
        # If the column is already a TimestampType (e.g. from Parquet), keep it;
        # otherwise parse using the supplied format string.
        col_dtype = dict(df.dtypes).get(snake)
        if col_dtype == "timestamp":
            pass  
        else:
            df = df.withColumn(snake, to_timestamp(col(snake), fmt))
    return df


# Build a single TimestampType column from separate integer or string
# year / month / day / [hour / minute / second] columns.
# Used for weather dataset
def assemble_timestamp_from_parts(df, config):
    if not config:
        return df

    year   = config["year_col"]
    month  = config["month_col"]
    day    = config["day_col"]
    hour   = config.get("hour_col")
    minute = config.get("minute_col")
    second = config.get("second_col")
    out    = config["output_col"]

    # Build a string like "2024-01-01 00:00:00" then parse it
    time_part = concat(
        lpad(col(hour).cast("string"),   2, "0") if hour   else lit("00"), lit(":"),
        lpad(col(minute).cast("string"), 2, "0") if minute else lit("00"), lit(":"),
        lpad(col(second).cast("string"), 2, "0") if second else lit("00"),
    )
    date_part = concat(
        col(year).cast("string"),  lit("-"),
        lpad(col(month).cast("string"), 2, "0"), lit("-"),
        lpad(col(day).cast("string"),   2, "0"),
    )
    datetime_str = concat(date_part, lit(" "), time_part)
    df = df.withColumn(out, to_timestamp(datetime_str, "yyyy-MM-dd HH:mm:ss"))
    return df

# Combine separate date-string and time-string columns into a single column
# Used for air_quality dataset
def normalize_date_time_pairs(df, pairs):
    for pair in pairs:
        date_col   = pair["date_col"]
        time_col   = pair["time_col"]
        output_col = pair["output_col"]
        fmt        = pair["format"]
        datetime_str = concat(col(date_col), lit(" "), col(time_col))
        df = df.withColumn(output_col, to_timestamp(datetime_str, fmt))
    return df


# Common data type normalisation

# Formatting (trim whitespace, empty places become NULL)
def normalize_string_columns(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            c = field.name
            df = df.withColumn(c, when(trim(col(c)) == "", None).otherwise(trim(col(c))))
    return df

# Replace boolean-like columns with actual boolean (Y / N become true and falls, etc...)
def normalize_boolean_columns(df, bool_cols):
    truthy = {"y", "yes", "true", "1"}
    falsy  = {"n", "no",  "false", "0"}

    for c in bool_cols:
        if c not in df.columns:
            continue
        lowered = lower(trim(col(c).cast("string")))
        df = df.withColumn(
            c,
            when(lowered.isin(*truthy), lit(True))
            .when(lowered.isin(*falsy),  lit(False))
            .otherwise(None)
            .cast(BooleanType())
        )
    return df

#Removes duplicates according to PK (or full row duplicates)
def perform_data_quality_checks(df, primary_keys):
    initial_count = df.count()

    if primary_keys:
        # Drop row if ANY part of the composite primary key is missing (Null)
        df = df.dropna(subset=primary_keys)

        # Drop duplicates based strictly on the combination of the primary keys
        df = df.dropDuplicates(subset=primary_keys)
    else:
        # If no primary key exists (like Taxi Trips), just drop exact full-row duplicates
        df = df.dropDuplicates()

    final_count = df.count()
    rejected_count = initial_count - final_count

    return df, initial_count, rejected_count


# For versioning in metadata
def schema_hash(df):
    return hash(tuple(sorted(df.dtypes)))


#-----------------------------Main Function-------------------
#steps follow what the assignment mentioned. 

def ingest_dataset(dataset_name, config):
    start_time = time.time()

    # Load according to file type (csv, parquet)
    df = spark.read.format(config["format"]).options(**config["options"]).load(config["path"])

    # Standardise column names (snake_case)
    df = standardize_columns(df)

    # cast each column to its expected type.
    df = apply_schema(df, config)

    # Validate input schema: column presence + type conformance
    validate_input_schema(df, config, dataset_name)

    # Normalise common data types (remove whitespace)
    df = normalize_string_columns(df)

    #  Normalise timestamps to yyyy-mm-dd HH:mm:ss
    df = normalize_timestamps(df, config.get("timestamp_cols", {}))
    #(weather)
    df = assemble_timestamp_from_parts(df, config.get("assembled_timestamp"))
    #(air quality)
    df = normalize_date_time_pairs(df, config.get("date_time_pairs", []))

    # Normalise boolean data types
    df = normalize_boolean_columns(df, config.get("bool_cols", []))

    # quality checks (removing duplicates)
    standard_pks = [to_snake_case(c) for c in config["primary_keys"]]
    df, initial_cnt, rejected_cnt = perform_data_quality_checks(df, standard_pks)

    # Write to Delta
    output_path = f"./delta/{dataset_name}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(output_path)


    execution_time = time.time() - start_time

    metadata = {
        "dataset":                dataset_name,
        "processed_records":      initial_cnt,
        "rejected_records":       rejected_cnt,
        "final_records":          initial_cnt - rejected_cnt,
        "execution_time_seconds": round(execution_time, 2),
        "schema_version":         schema_hash(df),
    }
    return metadata

#-----------------------------RUN framework-------------------
ingestion_logs = []
for name, conf in DATASET_CONFIGS.items():
    print(f"Ingesting {name}...")
    log = ingest_dataset(name, conf)
    ingestion_logs.append(log)

print(*ingestion_logs, sep="\n")


Ingesting weather...


Ingesting taxi_trips_01...


Ingesting taxi_trips_02...


Ingesting taxi_trips_03...


Ingesting taxi_zone_lookup...
Ingesting air_quality...


{'dataset': 'weather', 'processed_records': 8784, 'rejected_records': 0, 'final_records': 8784, 'execution_time_seconds': 4.41, 'schema_version': -6172388862951842371}
{'dataset': 'taxi_trips_01', 'processed_records': 2964624, 'rejected_records': 0, 'final_records': 2964624, 'execution_time_seconds': 15.78, 'schema_version': -3319613983096149512}
{'dataset': 'taxi_trips_02', 'processed_records': 3007526, 'rejected_records': 1, 'final_records': 3007525, 'execution_time_seconds': 19.24, 'schema_version': -3319613983096149512}
{'dataset': 'taxi_trips_03', 'processed_records': 3582628, 'rejected_records': 0, 'final_records': 3582628, 'execution_time_seconds': 17.05, 'schema_version': -8838179432208523880}
{'dataset': 'taxi_zone_lookup', 'processed_records': 265, 'rejected_records': 0, 'final_records': 265, 'execution_time_seconds': 4.28, 'schema_version': -6976494558679601782}
{'dataset': 'air_quality', 'processed_records': 8139551, 'rejected_records': 0, 'final_records': 8139551, 'executi

In [32]:
df_delta1 = spark.read.format("delta").load("delta/taxi_trips_01")
df_delta2 = spark.read.format("delta").load("delta/weather")
df_delta3 = spark.read.format("delta").load("delta/taxi_zone_lookup")
df_delta4 = spark.read.format("delta").load("delta/air_quality")

print("taxi_trips_01")
print(df_delta1.show(1))
print("weather")
df_delta2.show(1)
print("taxi_zone_lookup")
df_delta3.show(1)
print("air_quality")
df_delta4.show(1)

taxi_trips_01
+---------+--------------------+---------------------+---------------+-------------+-----------+------------------+--------------+--------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|vendor_id|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|ratecode_id|store_and_fwd_flag|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+---------+--------------------+---------------------+---------------+-------------+-----------+------------------+--------------+--------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|        2| 2024-01-01 00:38:58|  2024-01-01 00:53:37|            1.0|         1.94|          1|             false|           114|            79|          

# 5 - Build pipeline

Load taxi trip data, merge them in one table, then add weather data

In [ ]:
from pyspark.sql.functions import date_trunc

def rename_all(dataframe: DataFrame, prefix:str):
    """ Rename all columns of a dataframe by adding the given prefix"""
    for col_name in dataframe.columns:
        dataframe = dataframe.withColumnRenamed(col_name, prefix + col_name)
    return dataframe

taxi_trips_01 = spark.read.format("delta").load("delta/taxi_trips_01")
taxi_trips_02 = spark.read.format("delta").load("delta/taxi_trips_02")
taxi_trips_03 = spark.read.format("delta").load("delta/taxi_trips_03")

taxi_trips_all = taxi_trips_01.unionByName(taxi_trips_02).unionByName(taxi_trips_03)
weather = spark.read.format("delta").load("delta/weather")

# Add a prefix to make it clear where the data comes from after joining with taxi data
weather = rename_all(weather, "weather_")

integrated_taxi_trips = (
    taxi_trips_all
    .withColumn( 
        "weather_time",
        date_trunc("hour", col("tpep_pickup_datetime"))  # Add a column with the current hour of the pickup daytime, this is the one used for the join
    )
    .join(
        weather,
        on=col("weather_time") == col("weather_timestamp"),
        how="left"      # This ensures that by default, if there is no weather data at the current hour, taxi trip data is kept and weather is set to NULL
    )
    .drop("weather_timestamp")
)

integrated_taxi_trips.show(5)


26/09/11 11:39:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+--------------------+---------------------+---------------+-------------+-----------+------------------+--------------+--------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------------------+------------+-------------+-----------+------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+
|vendor_id|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|ratecode_id|store_and_fwd_flag|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|       weather_time|weather_year|weather_mont

Add pickup/dropoff borough/zone to the table

In [3]:
taxi_lookup = spark.read.format("delta").load("delta/taxi_zone_lookup")
taxi_lookup_bare = (
    taxi_lookup
        .drop("service_zone")
        .withColumnRenamed("borough", "pu_borough")
        .withColumnRenamed("zone", "pu_zone")
)

integrated_taxi_trips = (
    integrated_taxi_trips
        .join(
            taxi_lookup_bare,
            on=col("pu_location_id") == col("location_id"),
            how="left"  
        )
        .drop("location_id")
)

taxi_lookup_bare = (
    taxi_lookup_bare
        .withColumnRenamed("pu_borough", "do_borough")
        .withColumnRenamed("pu_zone", "do_zone")
)

integrated_taxi_trips = (
    integrated_taxi_trips
        .join(
            taxi_lookup_bare,
            on=col("do_location_id") == col("location_id"),
            how="left"  
        )
        .drop("location_id")
)
integrated_taxi_trips.show(5)


+---------+--------------------+---------------------+---------------+-------------+-----------+------------------+--------------+--------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------------------+------------+-------------+-----------+------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+----------+--------------------+----------+--------------------+
|vendor_id|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|ratecode_id|store_and_fwd_flag|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_sur

Add air quality data

In [4]:
air_quality = spark.read.format("delta").load("delta/air_quality")

# Only mesurments made in the state of New York (code 36) are kept.
air_quality_ny = air_quality.filter(col("state_code") == 36)

air_quality_ny = rename_all(air_quality_ny, "air_q_")

integrated_taxi_trips = (
    integrated_taxi_trips
        .join(
            air_quality_ny,
            on=(col("pu_borough") == col("air_q_county_name")) & (col("weather_time") == col("air_q_datetime_local")),
            how="left"
        )
        .drop("air_q_county_name")
        .drop("air_q_datetime_local")
)

integrated_taxi_trips.show(20)

+---------+--------------------+---------------------+---------------+-------------+-----------+------------------+--------------+--------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------------------+------------+-------------+-----------+------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+----------+--------------------+----------+-------------------+----------------+-----------------+--------------+--------------------+---------+--------------+---------------+-----------+--------------------+----------------+----------------+--------------+--------------+------------------------+----------------------+--

Save the final delta table in 2 versions: one partitioned by date and one partitioned by borough

In [6]:
integrated_taxi_trips = integrated_taxi_trips.withColumn("day_timestamp", date_trunc("day", "weather_time"))
# Partition by day
integrated_taxi_trips.write.format("delta").partitionBy("day_timestamp").save("delta/integrated_taxi_trips_by_date")
# Partition by borough
integrated_taxi_trips.write.format("delta").partitionBy("pu_borough").save("delta/integrated_taxi_trips_by_borough")

Benchmarks

In [ ]:
from pyspark.sql.functions import unix_seconds, from_unixtime

def execute_queries(dataframe:DataFrame):
    """
    Run the following queries:
    - number of taxi trips per borough (by convention the pickup borough is the one used for reference)
    - average trip duration per day,
    - average fare per borough.
    """
    results = []

    start = time.time()
    count = dataframe.select("pu_borough").groupBy("pu_borough").count()
    total_time = time.time() - start

    results.append((count, total_time))

    start = time.time()
    mean = (dataframe
            .withColumn("trip_time_seconds", 
                unix_seconds(col("tpep_dropoff_datetime")) - unix_seconds(col("tpep_pickup_datetime"))
            )
            .groupBy("day_timestamp")
            .avg("trip_time_seconds")
            .withColumn("trip_time", from_unixtime(col("avg(trip_time_seconds)").cast("int"), "HH:mm:ss"))
            .select("day_timestamp", "trip_time")
        )

    total_time = time.time() - start
    
    results.append((mean, total_time))

    start = time.time()
    mean = dataframe.select("pu_borough", "fare_amount").groupBy("pu_borough").avg("fare_amount")
    total_time = time.time() - start

    results.append((mean, total_time))

    return results

integrated_taxi_trips_by_date = spark.read.format("delta").load("delta/integrated_taxi_trips_by_date")
integrated_taxi_trips_by_borough = spark.read.format("delta").load("delta/integrated_taxi_trips_by_borough")

print("Partitioned by date:")

results_date = execute_queries(integrated_taxi_trips_by_date)

for i in range(len(results_date)):
    #results_date[i][0].show(5)
    print(f"Time (query {i+1}): {int(results_date[i][1]*1000)} ms")

print("\nPartitioned by borough:")

results_borough = execute_queries(integrated_taxi_trips_by_borough)

for i in range(len(results_borough)):
    #results_borough[i][0].show(5)
    print(f"Time (query {i+1}): {int(results_borough[i][1]*1000)} ms")



Partitioned by date:
Time (query 1): 14 ms
Time (query 2): 47 ms
Time (query 3): 18 ms

Partitioned by borough:
Time (query 1): 13 ms
Time (query 2): 45 ms
Time (query 3): 15 ms
